# Predicció del LLM generador de registres sintètics

## 1. Description

The dataset contains a mixture of demographic, clinical, laboratory, diagnosis, utilization, and discharge-related variables. Examples include age, BMI, blood pressure, heart rate, glucose, creatinine, diagnosis category, smoking status, number of medications, previous visits, and length of stay.

All records are fully synthetic and do not correspond to real patients.

The target variable contains four anonymized classes:
- Model_A
- Model_B
- Model_C
- Model_D

These correspond to four different LLMs used to generate the data.

For this challenge, the predictive model must be Logistic Regression. Participants are encouraged to improve performance through representation and preprocessing rather than by changing the classifier. Useful directions may include categorical encoding, missing-value strategies, scaling, binning, transformations, interaction features, and clinically motivated feature engineering.

## 2. Imports i configuració

In [27]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42


In [28]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="'penalty' was deprecated in version 1.8",
    category=FutureWarning
)

## 3. Carregar i inspeccionar les dades

In [29]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

X_raw = train.drop(columns=["id","label","discharge_destination"])
y = train["label"]
X_test_raw = test.drop(columns=["id","discharge_destination"])

print("Train:", train.shape)
print("Test:", test.shape)
print("Classes:", sorted(y.unique()))

display(train.head())
display(train.isna().sum().sort_values(ascending=False).head(10))


Train: (2400, 26)
Test: (1600, 25)
Classes: ['Model_A', 'Model_B', 'Model_C', 'Model_D']


,id,age,sex,bmi,smoking_status,encounter_type,systolic_bp,diastolic_bp,heart_rate,respiratory_rate,...,diabetes,hypertension,chronic_kidney_disease,primary_diagnosis,num_medications,prior_visits_12m,length_of_stay_days,discharge_destination,readmitted_30d,label
0,5be091fe7010722968d37fbbbfa5adec,38,Male,30.4,Current,Urgent,160,95,105,28,...,No,No,No,Respiratory,6,10,3.0,Home,No,Model_B
1,7ebc8ab8f1dfede1237163cb2e71325f,48,Female,28.5,Current,Elective,128,76,78,16,...,No,No,No,Gastrointestinal,3,2,2.0,Home,No,Model_D
2,69f51e35fec3bda1d81f2cbf9ac8b827,53,Female,25.4,Never,Elective,130,78,77,19,...,No,No,No,Renal,1,1,3.0,Rehabilitation,No,Model_A
3,8c64aa346b72cad6cb179f254abfd71d,45,Male,28.2,Never,Elective,145,95,72,18,...,Yes,No,No,Cardiovascular,4,0,4.0,Home,No,Model_C
4,c8255d88bcfc566c5616bed411b55bd7,41,Male,27.9,Current,Emergency,150,94,115,22,...,No,No,No,Gastrointestinal,2,5,3.0,HomeHealth,No,Model_B


creatinine_mg_dl         572
glucose_mg_dl            536
sodium_mmol_l            500
hemoglobin_g_dl          500
bmi                      268
readmitted_30d             0
discharge_destination      0
length_of_stay_days        0
prior_visits_12m           0
num_medications            0
dtype: int64

## 4. Feature engineering

Algo important a tenir en compte es la creació de _ld i _ex. Per exemple amb bmi tindrem:
- 'bmi': permet aprendre una relació contínua, per exemple, si els valors alts de BMI afavoreixen una classe.
- 'bmi_ex' : el valor exacte amb dos decimals passat a categòrica.  Permet aprendre per exemple que el valor exacte "30.4" és especialment freqüent en una classe.
-  'bmi_ld' : guardem l'ultim dígit. Permet aprendre que el decimal final "4" pot ser una empremta del model generador.

In [30]:
BINARY_COLUMNS = ["diabetes", "hypertension", "chronic_kidney_disease", "readmitted_30d"]
INTEGER_COLUMNS = ["age", "systolic_bp", "diastolic_bp", "heart_rate", "glucose_mg_dl", "sodium_mmol_l", "respiratory_rate", "spo2"]

def feature_engineering(data):
    data = data.copy()

    # yes no columns to 0 and 1
    for column in BINARY_COLUMNS:
        data[column] = data[column].map({"No": 0, "Yes": 1})

    #new features without dropping any columns
    data["mean_arterial_pressure"] = (data["systolic_bp"] + 2 * data["diastolic_bp"]) / 3
    data["bmi_age_interaction"] = data["bmi"] * data["age"]
    data["pulse_pressure"] = data["systolic_bp"] - data["diastolic_bp"]
    data["log_length"] = np.log1p(data["length_of_stay_days"])
    data["log_creatinine"] = np.log(data["creatinine_mg_dl"])

    # convert int and NaN to string
    def as_category(series):
        return series.astype("Int64").astype(str).where(series.notna(), "nan")

    # last digit of integer columns
    for column in INTEGER_COLUMNS:
        data[column + "_ld"] = as_category(data[column] % 10)
    
    for column in ["bmi", "temperature_c", "hemoglobin_g_dl", "length_of_stay_days"]:
        data[column + "_ld"] = as_category((data[column] * 10).round() % 10)
    data["creat_dec2"] = as_category((data["creatinine_mg_dl"] * 100).round() % 10)
    data["rr_parity"] = as_category(data["respiratory_rate"] % 2)

    # ara creem exact columns amb un máxim de 2 decimals
    exact_columns = ["temperature_c", "bmi", "hemoglobin_g_dl", "length_of_stay_days", "creatinine_mg_dl", "num_medications", "prior_visits_12m", *INTEGER_COLUMNS]
    for column in exact_columns:
        data[column + "_ex"] = data[column].round(2).astype(str).where(data[column].notna(), "nan")
    
    return data


## 5. Fingerprint interactions

Al inici vam començar amb l'objectiu d'eliminar las variables que no necessitàvem, tanmateix això va canviar perquè quan provàvem d'eliminar sempre baixàvem de puntuació, i cada vegada que afegíem variables noves semblava que la nostra accuracy anava pujant fins al punt que hem provat d'afegir aquest apartat.

La regressió logística és additiva, sense interacció entre variables només aprén l'efecte de cada variable de forma independent. Per capturar patrons conjunts, creem noves variables combinant dues variables existents i així veiem si es segueix algun patró conjunt.

Combinem les variables que hem trobat amb _ld i _ex que és com que semblen més de ML i les variables categòriques pures (aquelles que no son binàries).

In [31]:
def add_fingerprint_interactions(data):
    #comencem aplicant tot el feature engineering
    data = feature_engineering(data)

    
    fingerprint_columns = [
        column for column in data.columns
        if column.endswith("_ld") or column.endswith("_ex")
        or column in ["sex", "smoking_status", "encounter_type", "primary_diagnosis", "discharge_destination"]
    ]
    for index, first in enumerate(fingerprint_columns):
        for second in fingerprint_columns[index + 1:]:
            data[f"{first}__{second}"] = data[first].astype(str) + "|" + data[second].astype(str)
    return data

X = add_fingerprint_interactions(X_raw)
X_test = add_fingerprint_interactions(X_test_raw)
assert list(X.columns) == list(X_test.columns)
print("Variables abans de codificar:", X.shape[1])


Variables abans de codificar: 522


## 6. Preprocessament i model

Les variables numèriques s'imputen amb la mediana i s'estandarditzen. Les categòriques s'imputen amb la moda i es codifiquen amb one-hot encoding. Agrupem categories rares amb freqüència menor a 10 cops per evitar overfitting.

In [32]:
categorical_columns = X.select_dtypes(exclude=np.number).columns.tolist()
numeric_columns = [column for column in X.columns if column not in categorical_columns]

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), numeric_columns),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("encoder", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                        min_frequency=10,
                                                        sparse_output=True))]), categorical_columns),
])

classifier = OneVsRestClassifier(LogisticRegression(penalty=None, solver="lbfgs", max_iter=5000))
model = Pipeline([("preprocessor", preprocessor), ("classifier", classifier)])


## 7. Validació creuada

La validació és estratificada i mesura l'accuracy, que és la mètrica de la competició. Amb aquesta configuració s'espera aproximadament 0.95 d'accuracy local.

In [33]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
print("Accuracy per fold:", np.round(scores, 4))
print("Accuracy mitjana:", round(scores.mean(), 4))
print("Desviació estàndard:", round(scores.std(), 4))


Accuracy per fold: [0.9583 0.9542 0.9479 0.9542 0.9458]
Accuracy mitjana: 0.9521
Desviació estàndard: 0.0046


## 8. Entrenament final i submission

Entrenem amb totes les dades i guardem el fitxer amb les columnes requerides: `id` i `label`.

In [34]:
model.fit(X, y)
predictions = model.predict(X_test)
submission = pd.DataFrame({"id": test["id"], "label": predictions})
submission.to_csv("submissions/submission_final.csv", index=False)
print("Accuracy d'entrenament:", round(accuracy_score(y, model.predict(X)), 4))
print(submission["label"].value_counts())
display(submission.head())


Accuracy d'entrenament: 1.0
label
Model_B    423
Model_D    400
Model_C    396
Model_A    381
Name: count, dtype: int64


,id,label
0,0102f98c5fd7ed84754c58fe0d7f013f,Model_D
1,e2fa3e497f5bdb8c79748c7c0611fad0,Model_B
2,818a6ec782d097abc1ce67c42299d06a,Model_A
3,54e46a096cb47b5499a986477be83061,Model_B
4,5da9e7a746719fc3950c6e579c82bfa1,Model_C
